# 04 — Feature Analysis

Explore the extracted Paderborn feature table, inspect class and bearing distributions, analyse correlations and compare feature representations before model fitting.

In [ ]:
# ============================================================
# STAGE 4: FEATURE ANALYSIS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_DIR = Path("../data/processed")

feature_path = (
    PROCESSED_DIR /
    "paderborn_vibration_features.csv"
)

df = pd.read_csv(feature_path)

print("Dataset shape:", df.shape)

display(df.head())

In [ ]:
# ============================================================
# BASIC DATASET SUMMARY
# ============================================================

print("Rows:", len(df))
print("Unique bearings:", df["bearing"].nunique())
print("Unique conditions:", df["operating_condition"].nunique())

print("\nFault-location distribution:")
print(
    df["fault_location"].value_counts()
)

print("\nFault-class distribution:")
print(
    df["fault_class"]
    .value_counts()
    .sort_index()
)

print("\nBinary-label distribution:")
print(
    df["binary_label"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:", df.isna().sum().sum())

In [ ]:
# ============================================================
# DEFINE FEATURE COLUMNS
# ============================================================

metadata_columns = [
    "file",
    "bearing",
    "operating_condition",
    "measurement_number",
    "fault_location",
    "fault_class",
    "binary_label"
]

feature_columns = [
    column
    for column in df.columns
    if column not in metadata_columns
]

print("Number of numerical features:", len(feature_columns))
print("\nFeatures:")

for feature in feature_columns:
    print("-", feature)

In [ ]:
# ============================================================
# BEARING-LEVEL SUMMARY
# ============================================================

bearing_summary = (
    df.groupby(
        ["bearing", "fault_location", "fault_class"]
    )[feature_columns]
    .mean()
    .reset_index()
)

print(
    "Bearing-level summary shape:",
    bearing_summary.shape
)

display(bearing_summary.head())

In [ ]:
# ============================================================
# VERIFY BEARING-LEVEL CLASSES
# ============================================================

print(
    bearing_summary["fault_location"]
    .value_counts()
)

display(
    bearing_summary[
        ["bearing", "fault_location", "fault_class"]
    ]
)

In [ ]:
# ============================================================
# FEATURE CORRELATION
# DESCRIPTIVE ANALYSIS ONLY
# ============================================================

correlation_matrix = df[feature_columns].corr()

# Find highly correlated feature pairs
correlation_pairs = []

for i in range(len(feature_columns)):
    for j in range(i + 1, len(feature_columns)):

        f1 = feature_columns[i]
        f2 = feature_columns[j]

        corr = correlation_matrix.loc[f1, f2]

        if abs(corr) >= 0.90:
            correlation_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "correlation": corr
            })

high_corr_df = (
    pd.DataFrame(correlation_pairs)
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print(
    "Highly correlated feature pairs (|r| >= 0.90):"
)

display(high_corr_df)

In [ ]:
# ============================================================
# CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(16, 12))

plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.colorbar(
    label="Correlation"
)

plt.xticks(
    range(len(feature_columns)),
    feature_columns,
    rotation=90
)

plt.yticks(
    range(len(feature_columns)),
    feature_columns
)

plt.title("Correlation Matrix of Extracted Vibration Features")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FEATURE MEANS BY FAULT LOCATION
# ============================================================

feature_means_by_class = (
    df.groupby("fault_location")[feature_columns]
      .mean()
      .T
)

display(
    feature_means_by_class
)

In [ ]:
# ============================================================
# BEARING-LEVEL FEATURE SUMMARY
# ============================================================

bearing_feature_means = (
    df.groupby(
        ["bearing", "fault_location", "fault_class"]
    )[feature_columns]
    .mean()
    .reset_index()
)

print(
    "Bearing-level feature matrix:",
    bearing_feature_means.shape
)

display(
    bearing_feature_means.head()
)

In [ ]:
# ============================================================
# WITHIN-BEARING FEATURE VARIATION
# ============================================================

bearing_feature_std = (
    df.groupby("bearing")[feature_columns]
      .std()
)

display(
    bearing_feature_std.describe().T
)